# Занятие 6, пара 1. Группировка и объединение в pandas


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## От отдельных строк к ответу на вопрос

В прошлый раз мы разбирались, каким данным можно доверять. Теперь хочется спросить таблицу:
«Где слушали больше музыки? А какой жанр?» Таблица большая или маленькая — вопрос всё равно
должен иметь понятный смысл. **Событий больше** и **минут больше** — разные утверждения.

Сначала вручную соберём одну группу. Потом поручим ту же работу `groupby`. После этого
подтянем названия и жанры из справочника и поймаем ошибку, из-за которой отчёт внезапно «растёт».

Нужны: выбор столбцов, `.loc`, маски и пропуски из блоков 02 и 05. Новые операции разберём здесь.
После пары вы сможете посчитать показатель по группам, различить число строк и известных значений,
присоединить справочник без размножения событий и объяснить, какие данные не попали в итог.
Результат — сводная таблица, список несопоставленных ключей и короткий вывод.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Посмотрим на исходник

Импортируем знакомый Pandas; `display` показывает таблицу в Jupyter. Укажем текстовый тип
ключей при чтении, чтобы `001` не превратился в `1`. Пустое поле длительности станет пропуском.
Набор намеренно небольшой: любой важный результат можно проверить глазами.


In [ ]:
import pandas as pd
from IPython.display import display

listens = pd.read_csv(DATA_DIR / 'aggregation' / 'listens.csv', dtype={'event_id': 'string', 'track_id': 'string'})
tracks = pd.read_csv(DATA_DIR / 'aggregation' / 'tracks.csv', dtype={'track_id': 'string'})
display(listens)
display(tracks)


У нас 12 событий. Повтор `track_id` нормален: одну песню могут слушать несколько раз.
Повтор `event_id` был бы другой историей — одна запись события попала дважды. Поэтому прежде
чем бороться с дублями, надо назвать ключ. Паспорт набора лежит в `aggregation/SOURCE.md`.

**До запуска:** сколько строк с площадкой «Север»? Все ли их минуты известны?


In [ ]:
north_mask = listens['city'] == 'Север'
north = listens.loc[north_mask]
display(north)
north_minutes = north['minutes'].sum()
print('Известные минуты на Севере:', north_minutes)


### 2. А теперь то же для каждой группы

Повторять фильтр для каждого города неудобно: завтра появится третий. `groupby('city')`
разделяет строки по одинаковому значению города. Пока это **описание группировки**, не итоговая
таблица. Выбираем столбец `minutes`, затем `.sum()` складывает его значения внутри каждой группы.
Результат — Series: индекс содержит города, значения — суммы известных минут.


In [ ]:
by_city = listens.groupby('city')
city_minutes = by_city['minutes'].sum()
display(city_minutes)


На Севере 19, на Юге 21 известная минута. Это ещё не значит «средний человек на Юге слушает
дольше»: людей мы вообще не считали, а часть длительностей неизвестна.

`.size()` считает **строки группы**, `.count()` — **заполненные значения выбранного столбца**.
Ноль заполнен, поэтому входит в count. Пропуск не входит. Проверим разницу до расчёта среднего.


In [ ]:
city_events = by_city.size()
city_known = by_city['minutes'].count()
display(city_events)
display(city_known)
city_mean = by_city['minutes'].mean()
display(city_mean)


На Юге 6 событий, но только 4 известные длительности. Среднее — 21 / 4, а не 21 / 6.
Автоматически заменять неизвестные минуты нулём ради удобного деления нельзя.

Есть ещё одна ловушка: обычная сумма пропусков может выглядеть как ноль. Параметр
`min_count=1` требует хотя бы одно известное значение; без него результат останется пропуском.


In [ ]:
unknown_minutes = pd.Series([None, None], dtype='float64')
print('Обычная сумма:', unknown_minutes.sum())
print('Требуем известное значение:', unknown_minutes.sum(min_count=1))


### ✏️ Ваш ход — не потерять знаменатель

Получите `my_sizes` — число событий по городам, `my_known` — число
известных длительностей и `my_means` — среднее известных минут. Используйте исходную `listens`.
Объясните, почему одинаковое число событий не означает одинаковый знаменатель среднего.
Не заполняйте пропуски. У вас 7 минут на код и 3 на обсуждение.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.1.1', my_sizes, my_known, my_means)


**Мой вывод:** …


### 3. Несколько показателей в одной таблице

Чтобы не держать три результата порознь, используем `.agg(...)` — задать агрегаты.
В именованной записи `events=('event_id', 'size')` слева имя **нового** столбца,
в скобках — **исходный** столбец и операция. Сначала соберём только два показателя.
Это не новая формула: size и count мы уже проверили.


In [ ]:
city_report = by_city.agg(
    events=('event_id', 'size'),
    known_minutes=('minutes', 'count')
)
display(city_report)


Добавим к готовой таблице знакомые суммы и средние. Pandas сопоставляет Series по индексу
города, а не просто «первую строку с первой». `.reset_index()` превратит названия городов из
индекса в обычный столбец. Это удобно для сохранения и дальнейшего соединения.


In [ ]:
city_report['total_minutes'] = by_city['minutes'].sum(min_count=1)
city_report['mean_minutes'] = by_city['minutes'].mean()
city_report = city_report.reset_index()
display(city_report)


### 4. Где взять жанр?

Жанр хранится в `tracks`, а события — в `listens`. Соединять их по номеру строки нельзя:
порядок строк ничего не говорит о том, какой трек слушали. Общий смысл есть у `track_id`.

`merge` ищет совпадение **значений ключа**. `on='track_id'` задаёт ключ; `how='left'` сохраняет
все события слева. Если справа совпадения нет, новые поля будут пустыми.
`validate='many_to_one'` проверяет обещание: слева ключ может повторяться, справа — уникален.
`indicator=True` добавит `_merge`, чтобы видно было, нашлась ли пара. Проверим обе исходные таблицы.


In [ ]:
display(listens[['event_id', 'track_id']])
display(tracks)
enriched = listens.merge(tracks, on='track_id', how='left', validate='many_to_one', indicator=True)
display(enriched)


Для `t99` названия и жанра нет. Это не повод придумывать жанр и не доказательство,
что событие ошибочно: возможно, справочник неполон. `left_only` означает «осталась только
левая строка», `both` — «нашлась пара». Выделим неизвестное явно.


In [ ]:
unmatched_mask = enriched['_merge'] == 'left_only'
unmatched = enriched.loc[unmatched_mask, ['event_id', 'track_id', 'minutes']]
display(unmatched)
print('До:', len(listens), 'после:', len(enriched))


Для сравнения сделаем `inner`: сохраняются только совпавшие ключи. Получим 11 событий,
и потеряем 7 известных минут. `inner` не плохой сам по себе; он отвечает другому условию:
«только события, для которых есть справочник». `outer` сохранил бы ещё и правый `t05` без событий.
Для нашего отчёта по всем событиям выбираем left.


In [ ]:
matched_only = listens.merge(tracks, on='track_id', how='inner', validate='many_to_one')
print('Все известные минуты:', listens['minutes'].sum())
print('После inner:', matched_only['minutes'].sum())


### ✏️ Ваш ход — найти потерянную запись

Из готовой основной таблицы `enriched` получите `my_missing_ids` —
Series ключей событий без найденного жанра. Посчитайте их известные минуты в `my_missing_minutes`.
Запишите, какой вопрос вы задали бы владельцу справочника. Используйте `_merge`, `.loc`, `.sum()`.
Автопроверка проверит только числа и ключ, ваш вопрос обсудим отдельно.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.1.2', my_missing_ids, my_missing_minutes)


**Мой вывод:** …


### 5. Почему после объединения строк стало больше?

Создадим маленькую ошибку отдельно: добавим вторую копию первой строки справочника.
`pd.concat([таблица 1, таблица 2], ignore_index=True)` ставит строки друг под другом и создаёт
новый последовательный индекс. В отличие от merge, concat не ищет пары по ключу.
`.iloc[[0]]` выбирает первую строку **по позиции**, сохраняя её таблицей.


In [ ]:
first_track = tracks.iloc[[0]]
bad_tracks = pd.concat([tracks, first_track], ignore_index=True)
bad_join = listens.merge(bad_tracks, on='track_id', how='left')
print('Было событий:', len(listens), 'получилось строк:', len(bad_join))
print('Было минут:', listens['minutes'].sum(), 'стало:', bad_join['minutes'].sum())


Три события t01 встретили по две подходящие строки. Получилось 15 строк и 48 минут,
хотя никто дополнительно музыку не слушал. Merge честно перечислил все пары.
Не лечим результат удалением случайных строк: сначала проверяем справочник.
Ниже намеренная ошибка изолирована: ловим только ожидаемый `MergeError`, чтобы тетрадь продолжилась.


In [ ]:
try:
    listens.merge(bad_tracks, on='track_id', how='left', validate='many_to_one')
except pd.errors.MergeError as error:
    print('Проверка остановила неоднозначный справочник:', error)


### ✏️ Ваш ход — проверить ключ до соединения

С помощью знакомого `.duplicated(keep=False)` найдите
в `bad_tracks` все строки с повторяющимся `track_id`. Сохраните таблицу в `my_duplicates`.
Посчитайте `my_extra_rows`: сколько лишних строк появилось в `bad_join` по сравнению с `listens`.
Почему два повтора ключа в справочнике затронули три события?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.1.3', my_duplicates, my_extra_rows)


**Мой вывод:** …


### 6. Финальный отчёт: неизвестное тоже должно быть видно

Вернёмся к корректному `enriched`, а не к упражнению и не к `bad_join`. По умолчанию groupby
не включает пустой ключ группы. `dropna=False` сохраняет отдельную группу для неизвестного жанра.
Суммируем только известные минуты; если вся группа неизвестна, `min_count=1` сохранит пропуск.
`.sort_values(ascending=False)` расположит суммы от большей к меньшей.


In [ ]:
genre_groups = enriched.groupby('genre', dropna=False)
genre_minutes = genre_groups['minutes'].sum(min_count=1)
genre_minutes = genre_minutes.sort_values(ascending=False)
display(genre_minutes)
print('Минут в сводке:', genre_minutes.sum())
print('Минут в исходнике:', listens['minutes'].sum())


Суммы сошлись: 40 минут. Это полезная проверка, но не доказательство правильности каждого
жанра. Напишите две фразы: что лидирует **по известному времени** и какую неопределённость мы
сохранили. Ничего не утверждаем о популярности среди людей — данных о людях здесь нет.

Сохраним отчёт отдельно от учебного исходника. `mkdir(..., exist_ok=True)` создаёт папку,
если её нет. Имя файла выберите свободное, если там уже лежит ваша работа.


In [ ]:
report_dir = WORKSPACE_DIR / 'reports'
report_dir.mkdir(exist_ok=True)
report_path = report_dir / '06_city_report.csv'
city_report.to_csv(report_path, index=False)
print('Сохранено:', report_path)


### Ещё один опыт, если осталось время: сводная таблица

Хотим видеть города строками, жанры столбцами. `pivot_table` меняет форму сводки:
`index` — строки, `columns` — столбцы, `values` — измерение, `aggfunc` — способ его объединить.
Сначала явно подпишем неизвестный жанр в **копии**. В сводной считаем события, поэтому
отсутствующее сочетание действительно даёт 0. Для неизвестной длительности такое заполнение неверно.


In [ ]:
pivot_source = enriched.copy()
pivot_source['genre'] = pivot_source['genre'].fillna('неизвестно')
pivot = pivot_source.pivot_table(index='city', columns='genre', values='event_id', aggfunc='count', fill_value=0)
display(pivot)


**Опыт на 10 минут:** предскажите три клетки вручную. Сравните сумму всех клеток с 12.
Затем замените счёт событий на среднее минут и уберите `fill_value=0`. Объясните, почему
пустые клетки теперь нельзя автоматически читать как нулевое прослушивание. Это расширение,
переменные из него не нужны следующей тетради.

Почитать по желанию: [Уэс Маккинни, Python for Data Analysis, глава о группировках](https://wesmckinney.com/book/data-aggregation).
Справочники: [groupby](https://pandas.pydata.org/docs/user_guide/groupby.html),
[merge и проверка связей](https://pandas.pydata.org/docs/user_guide/merging.html). Не нужно читать всё сразу:
найдите пример одного метода, который сегодня пригодился.
